# 04 — Agent Tools as MCP Functions

**Purpose:** Register the RAG answerer, analysis-report generator, and classifier as plain
Unity Catalog SQL functions in one schema, so Databricks' managed MCP server automatically
exposes all three at a single endpoint — no custom MCP server process needed.

### What This Notebook Does
1. Creates `ask_insurance_rag(question)` — calls the Mosaic AI Model Serving RAG endpoint
   from `src/03_rag_chain_and_deploy` via `ai_query()`.
2. Creates `generate_analysis_report(region, smoker)` — aggregates `docs.policy_documents`
   and writes a narrative report via `ai_gen()`.
3. Creates `classify_customer(description)` — classifies a free-text applicant description
   into a cost-risk tier via `ai_classify()`.
4. Validates all three, then prints the managed MCP endpoint URL.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | `insurance_rag_agent.docs.policy_documents`, `insurance_rag_endpoint` |
| Target | `insurance_rag_agent.agent_tools.*` (3 UC functions) |

> **Prerequisites:** Run `src/03_rag_chain_and_deploy` first (the RAG endpoint must exist
> for `ask_insurance_rag` to have something to call).

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG              = 'insurance_rag_agent'
AGENT_TOOLS_SCHEMA    = 'agent_tools'
DOCS_TABLE            = f'{CATALOG}.docs.policy_documents'
RAG_ENDPOINT_NAME     = 'insurance_rag_endpoint'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{AGENT_TOOLS_SCHEMA}')

print(f'Schema: {CATALOG}.{AGENT_TOOLS_SCHEMA}')
print(f'RAG endpoint: {RAG_ENDPOINT_NAME}')

---

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.ask_insurance_rag(
  question STRING COMMENT 'A natural-language question about the insurance policy data'
)
RETURNS STRING
COMMENT 'Answers a question about the insurance policy data using the deployed Mosaic AI Model Serving RAG endpoint.'
RETURN ai_query('insurance_rag_endpoint', question)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_analysis_report(
  p_region STRING COMMENT 'Region to filter by (northeast/northwest/southeast/southwest), or NULL for all regions',
  p_smoker STRING COMMENT 'Smoker status to filter by (yes/no), or NULL for all statuses'
)
RETURNS STRING
COMMENT 'Aggregates the insurance policy data by region/smoker status and writes a short narrative analysis report.'
RETURN (
  SELECT ai_gen(
    concat(
      'Write a concise 3-4 sentence business analysis report from these insurance policy statistics: ',
      'segment=', coalesce(p_region, 'all regions'), '/', coalesce(p_smoker, 'all smoker statuses'), ', ',
      'policy_count=', cast(count(*) AS STRING), ', ',
      'avg_charges=', cast(round(avg(charges), 2) AS STRING), ', ',
      'avg_bmi=', cast(round(avg(bmi), 2) AS STRING), ', ',
      'avg_age=', cast(round(avg(age), 1) AS STRING), '.'
    )
  )
  FROM insurance_rag_agent.docs.policy_documents
  WHERE (p_region IS NULL OR region = p_region)
    AND (p_smoker IS NULL OR smoker = p_smoker)
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.classify_customer(
  description STRING COMMENT 'A free-text description of a prospective customer (age, smoker status, BMI, region, etc.)'
)
RETURNS STRING
COMMENT 'Classifies a prospective customer description into a Low/Medium/High insurance cost-risk tier.'
RETURN ai_classify(description, ARRAY('Low Risk', 'Medium Risk', 'High Risk'))

---

In [0]:
%sql
SELECT
  insurance_rag_agent.agent_tools.classify_customer(
    '52-year-old smoker from the southeast region, BMI 34, 2 children'
  ) AS sample_classification,
  insurance_rag_agent.agent_tools.generate_analysis_report('southeast', 'yes') AS sample_report

In [0]:
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
mcp_url       = f'https://{workspace_url}/api/2.0/mcp/functions/{CATALOG}/{AGENT_TOOLS_SCHEMA}'

print(f'Managed MCP endpoint for these tools: {mcp_url}')